## ChromaDB Learning

[Documentation Link](https://docs.trychroma.com/docs/collections/manage-collections)

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture
# ! pip install chromadb

initializing directories

In [3]:
import os

In [4]:
ingestion_doc_path = os.path.join(os.getcwd(), 'ingestion_data')
chroma_db_persist_dir = os.path.join(os.getcwd(), 'chroma_store')

In [5]:
import chromadb

In [6]:
# creating chromadb client
# chroma_client = chromadb.Client() # In Memory Client
chroma_persistent_client = chromadb.PersistentClient(path=chroma_db_persist_dir)

ChromaDB Collections:

Collections are where we'll store our embeddings, documents, and any additional metadata. Collections index our embeddings and documents, and enable efficient retrieval and filtering. You can create a collection with a name:

In [7]:
# creating collection
collection_name = "heros_collection"

if collection_name in [item.name for item in chroma_persistent_client.list_collections()]:
  chroma_persistent_client.delete_collection(name=collection_name)

heros_collection = chroma_persistent_client.create_collection(name=collection_name)

print("document count:", heros_collection.count())


document count: 0


Chroma Collection - add: Add method adds a new required despite of being duplciate

Chroma Collection - upsert: Upsert method update in case exists otherwise inserts.

Let's first get the document and prepare the metadata for them

In [8]:
import os
import random

In [9]:
all_txt_files = os.listdir(ingestion_doc_path)

documents_details_list = []
counter = 0
for file_name in all_txt_files:
  counter+=1
  document_details_dict = {}
  document_details_dict["id"] = "doc_" + str(counter)
  with open(os.path.join(ingestion_doc_path, file_name)) as tFile:
    document_details_dict["content"] = tFile.read()

  document_details_dict["metadata"] = {
      "doc_id": document_details_dict["id"],
      "doc_name": file_name.split(".")[0],
      "doc_full_name": file_name,
      "doc_content_length": len(document_details_dict["content"]),
      "superstar_rating": random.randint(6,10)
  }

  # now adding these into main list:
  documents_details_list.append(document_details_dict)

print("length of document list: ", len(documents_details_list) )

length of document list:  19


Now, let's upsert into collection

In [10]:
# before upserting let's split the info for document

documents =  [item["content"] for item in documents_details_list]
ids =  [item["id"] for item in documents_details_list]
metadatas =  [item["metadata"] for item in documents_details_list]

In [11]:
# let's add into collection, it will create the embeddings. if we give "embeddings" paramter then it will skip creating embeddings.

heros_collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadatas
)

print("document count", heros_collection.count())

document count 19


Perfect! let's query it

In [12]:
result = heros_collection.query(
    query_texts = ["who holds Mjolnir"],
    n_results = 2
)
result

{'ids': [['doc_17', 'doc_11']],
 'embeddings': None,
 'documents': [['The God of Thunder: Thor Odinson, Heir of Asgard\nThe legend begins in the celestial realm of Asgard, one of the Nine Realms, where Thor Odinson was born the crown prince, son of Odin Allfather, the ruler of Asgard, and Gaea, the Elder Goddess of the Earth. From his earliest days, Thor was characterized by boundless strength, fiery courage, and, often, a crippling arrogance. He was trained relentlessly in the arts of combat, becoming the mightiest warrior in Asgard. His defining symbol and tool is Mjolnir, the enchanted uru hammer, forged in the heart of a dying star and imbued with Odin\'s powerful enchantment: "Whosoever holds this hammer, if he be worthy, shall possess the power of Thor." It was this enchantment that would define his entire existence. Due to his growing impulsiveness and hubris, which threatened to plunge Asgard into war, Odin eventually banished Thor to Earth (Midgard), stripped of his memory and

let's look into metadata

In [13]:
result["metadatas"]

[[{'doc_name': 'thor',
   'doc_full_name': 'thor.txt',
   'doc_content_length': 6151,
   'doc_id': 'doc_17',
   'superstar_rating': 7},
  {'doc_name': 'loki',
   'superstar_rating': 10,
   'doc_full_name': 'loki.txt',
   'doc_content_length': 5855,
   'doc_id': 'doc_11'}]]

Try updating the metadata

In [15]:
# let's update the loki rating, but before that let's grab it
loki_metadata = [heros for heros in result["metadatas"][0] if heros["doc_name"] == "loki"][0]
loki_metadata

{'doc_name': 'loki',
 'superstar_rating': 10,
 'doc_full_name': 'loki.txt',
 'doc_content_length': 5855,
 'doc_id': 'doc_11'}

In [16]:
# letn's update the rating
heros_collection.update(
    ids=[loki_metadata['doc_id']],
    metadatas=[{"superstar_rating":100}]
)
print("document updated")

document updated


Perfect! Let's ask again and look into only metadata

In [17]:
heros_collection.query(
    query_texts = ["who holds Mjolnir"],
    n_results = 2
)["metadatas"]

[[{'doc_content_length': 6151,
   'doc_full_name': 'thor.txt',
   'doc_name': 'thor',
   'superstar_rating': 7,
   'doc_id': 'doc_17'},
  {'superstar_rating': 100,
   'doc_full_name': 'loki.txt',
   'doc_content_length': 5855,
   'doc_id': 'doc_11',
   'doc_name': 'loki'}]]

Bingooo!!!! Could see superstar rating updated!!!

#### Trying Some other operations

In [18]:
# adding another record into collection

heros_collection.add(
    ids=['doc404'],
    documents=['This is error document'],
    metadatas=[{
        "doc_id":"doc404",
        "doc_name": 'error',
        "doc_full_name": 'error.txt',
        "doc_content_length": 30,
        "superstar_rating": 0
    }]
)

print("document added into collection:- ", heros_collection.count())

document added into collection:-  20


get the document with where clause

In [19]:
heros_collection.query(
    query_texts=["this is error document"],
    n_results=1
)["metadatas"]

[[{'superstar_rating': 0,
   'doc_content_length': 30,
   'doc_full_name': 'error.txt',
   'doc_id': 'doc404',
   'doc_name': 'error'}]]

let's update the document content

In [20]:
new_content = "The error has been addressed. Now this could be the rising hero."

heros_collection.update(
    ids=['doc404'],
    documents=[new_content],
    metadatas=[{'doc_content_length':80}]
)

print("document has updated")

document has updated


Let's view the updated document

In [21]:
heros_collection.get('doc404')

{'ids': ['doc404'],
 'embeddings': None,
 'documents': ['The error has been addressed. Now this could be the rising hero.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'doc_full_name': 'error.txt',
   'doc_name': 'error',
   'doc_id': 'doc404',
   'superstar_rating': 0,
   'doc_content_length': 80}]}

Nice, its updated